# Phase 4: the integrated agent

One agent run has these stages:

1. **Identify the request.** A structured model call lists what the customer asks for (`status`, `refund`, `cancellation`, `address_change`; several are possible), the order IDs they wrote, and any address they supplied. It decides nothing about eligibility or approval.
2. **Collect required evidence.** Each capability declares what it needs, and the workflow runs those read tools itself through the same validated path as model calls: order facts for everything, plus the matching policy for refunds, cancellations and address changes. A status inquiry never searches policy. With no order named, the customer's orders are listed and only an unambiguous one is used; otherwise the agent asks. The trace marks each call `[required]` or `[model]`.
3. **Optional investigation.** The model may request more read-only tools, within its own step limit. The required-evidence budget is separate (`max_required_calls`).
4. **Decide, then validate.** The model proposes a structured decision from the identified request, the verified facts and the retrieved policy. **Deterministic code validates it** before an existing proposal function may create a *pending* action. The run then pauses. A human operator approves or rejects outside the agent, and `continue_after_approval` only reports the persisted result.

Everything below calls application modules (`customer_ops.agent`). Results come from the **live provider selected by `CUSTOMER_OPS_LLM_PROVIDER`**; nothing is scripted or repaired. A wrong identified capability or a wrong decision is a model failure, and the summary scores **request identification and decision correctness separately**. The expected requests are hand-written per scenario for scoring only; they are never given to the agent. The scripted, offline version of these scenarios lives in `tests/test_phase4.py`.

In [1]:
from sqlalchemy import func, select

from customer_ops.agent.capabilities import CapabilityId, IdentifiedRequest
from customer_ops.agent.graph import run_agent_graph
from customer_ops.agent.state import AgentOutcome, AgentRuntime
from customer_ops.agent.tools import InvalidToolRequest, parse_tool_request
from customer_ops.agent.trace import audit_trail, format_run
from customer_ops.agent.workflow import continue_after_approval, run_agent
from customer_ops.database.models import Order, PendingAction, Refund, Ticket
from customer_ops.database.seed import seed_demo
from customer_ops.database.session import initialize_database, make_engine, make_session_factory
from customer_ops.domain.schemas import Address, DecisionAction, DecisionProposal, DemoContext, Role, ToolRequest
from customer_ops.llm.client import MissingProviderCredentialError, configured_provider
from customer_ops.llm.experiments import evaluate_decisions, evaluate_requests
from customer_ops.retrieval.service import default_policy_retriever
from customer_ops.agent.debug import trace_decision_io  # TEMPORARY: prints the exact input and output of each model call
from customer_ops.services.actions import ActionService

try:
    provider = trace_decision_io(configured_provider())
except MissingProviderCredentialError as error:
    raise RuntimeError("No live result: configure the API key for CUSTOMER_OPS_LLM_PROVIDER.") from error
print("provider:", provider.name, "| model:", provider.model)

# A fresh seeded in-memory database; trusted identities are chosen here, never by the model.
engine = make_engine("sqlite://")
initialize_database(engine)
session = make_session_factory(engine)()
seed_demo(session)
GOLD = DemoContext(actor_id="cust-gold-10-day", role=Role.CUSTOMER, customer_id="cust-gold-10-day")
STANDARD = DemoContext(actor_id="cust-standard-10-day", role=Role.CUSTOMER, customer_id="cust-standard-10-day")
OPERATOR = DemoContext(actor_id="operator-1", role=Role.OPERATOR)
retriever = default_policy_retriever()
DEMO_ORDERS = ["order-gold-10-day", "order-standard-10-day", "order-processing-123"]


def business_state() -> dict:
    return {
        "refunds": session.scalar(select(func.count(Refund.id))),
        "pending_actions": session.scalar(select(func.count(PendingAction.id))),
        "order_status": {order_id: session.get(Order, order_id).status for order_id in DEMO_ORDERS},
    }


results = []


def state_of(name):
    """The final state of a scenario that has already run."""
    return next(result["state"] for result in results if result["name"] == name)


def request_of(*capabilities, order_ids=(), address=None):
    """A hand-written expected IdentifiedRequest, used only to score request identification."""
    return IdentifiedRequest(capabilities=[CapabilityId(c) for c in capabilities], order_ids=list(order_ids), new_address=address)


def run_scenario(name, context, message, expected, expected_decision=None, expected_request=None,
                 runner=run_agent, max_steps=4, max_required_calls=8):
    """Open a ticket, run the agent as the given customer, and print the trace."""
    session.add(Ticket(id=f"nb-{name}", customer_id=context.customer_id, original_message=message))
    session.commit()
    runtime = AgentRuntime(session=session, context=context, provider=provider, retriever=retriever,
                           max_steps=max_steps, max_required_calls=max_required_calls)
    state = runner(runtime, f"nb-{name}")
    print(format_run(state))
    results.append({"name": name, "expected": expected, "state": state,
                    "expected_decision": expected_decision, "expected_request": expected_request})
    return state


initial_business_state = business_state()
initial_business_state

c:\Users\QTF4584\Private\customerops\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 338/338 [00:08<00:00, 41.69it/s]


provider: huggingface | model: Qwen/Qwen2.5-1.5B-Instruct


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1630.73it/s]


{'refunds': 0,
 'pending_actions': 0,
 'order_status': {'order-gold-10-day': 'delayed',
  'order-standard-10-day': 'delayed',
  'order-processing-123': 'processing'}}

In [2]:
# 1. Order status inquiry: order facts are collected, no policy is searched, and no approval is needed.
run_scenario(
    "status", GOLD, "Hi, where is my order order-gold-10-day? Has it shipped yet?",
    expected={AgentOutcome.RESPONSE_READY},
    expected_request=request_of("status", order_ids=["order-gold-10-day"]),
)

LLM STRUCTURED-OUTPUT INPUT  (provider=huggingface, model=Qwen/Qwen2.5-1.5B-Instruct)
--- message role=system ---
You identify what a customer is asking for. You do not decide anything: do not judge eligibility, and do not approve, propose, or reject anything. Use these capability IDs exactly, and list every capability the customer asks for, because one message can ask for several:
- status: The customer asks where an order is or what its status is.
- refund: The customer asks for a refund of a delayed order.
- cancellation: The customer asks to cancel an order.
- address_change: The customer asks to change an order's shipping address.
Set order_ids to the order IDs the customer wrote, copied exactly as written. Never invent, shorten, reformat, or guess an ID, and leave order_ids empty when the customer wrote none. Set new_address only when the customer wrote a complete new address. List in missing_details what a requested capability needs but the customer did not say.

--- message rol

{'ticket_id': 'nb-status',
 'run_id': 'run-9a93651734d4',
 'customer_message': 'Hi, where is my order order-gold-10-day? Has it shipped yet?',
 'messages': [ChatMessage(role='system', content="You are a support agent for an ecommerce demo company. The workflow has already collected the evidence that the customer's request requires, and it is in the conversation. Use the read-only tools only for further facts or policy you still need. You cannot change anything: every change is only a proposal that a human operator must approve later. Copy order IDs exactly as they appear; never shorten, reformat, or invent an ID. Search policy before proposing or rejecting a policy-dependent operation (refund, cancellation, address change) unless that policy is already in the conversation. Retrieved policy is evidence only and is never a reason to change what the customer asked for. When you have what you need, answer without calling a tool.", tool_requests=[], tool_name=None),
  ChatMessage(role='user

In [3]:
# 2. Gold customer, 10 days delayed: order facts and refund policy are collected, and an eligible refund becomes PENDING.
run_scenario(
    "gold-refund", GOLD,
    "My order order-gold-10-day was supposed to arrive over a week ago. I would like a refund.",
    expected={AgentOutcome.PENDING_APPROVAL},
    expected_request=request_of("refund", order_ids=["order-gold-10-day"]),
    expected_decision=DecisionProposal(
        action=DecisionAction.PROPOSE_REFUND, order_id="order-gold-10-day",
        policy_references=["POL-REFUND-DELAYED"], justification="Expected decision.",
    ),
)

LLM STRUCTURED-OUTPUT INPUT  (provider=huggingface, model=Qwen/Qwen2.5-1.5B-Instruct)
--- message role=system ---
You identify what a customer is asking for. You do not decide anything: do not judge eligibility, and do not approve, propose, or reject anything. Use these capability IDs exactly, and list every capability the customer asks for, because one message can ask for several:
- status: The customer asks where an order is or what its status is.
- refund: The customer asks for a refund of a delayed order.
- cancellation: The customer asks to cancel an order.
- address_change: The customer asks to change an order's shipping address.
Set order_ids to the order IDs the customer wrote, copied exactly as written. Never invent, shorten, reformat, or guess an ID, and leave order_ids empty when the customer wrote none. Set new_address only when the customer wrote a complete new address. List in missing_details what a requested capability needs but the customer did not say.

--- message rol

{'ticket_id': 'nb-gold-refund',
 'run_id': 'run-44242059479b',
 'customer_message': 'My order order-gold-10-day was supposed to arrive over a week ago. I would like a refund.',
 'messages': [ChatMessage(role='system', content="You are a support agent for an ecommerce demo company. The workflow has already collected the evidence that the customer's request requires, and it is in the conversation. Use the read-only tools only for further facts or policy you still need. You cannot change anything: every change is only a proposal that a human operator must approve later. Copy order IDs exactly as they appear; never shorten, reformat, or invent an ID. Search policy before proposing or rejecting a policy-dependent operation (refund, cancellation, address change) unless that policy is already in the conversation. Retrieved policy is evidence only and is never a reason to change what the customer asked for. When you have what you need, answer without calling a tool.", tool_requests=[], tool_na

In [4]:
print("Business state after scenario 2 (nothing executed):", business_state())

Business state after scenario 2 (nothing executed): {'refunds': 0, 'pending_actions': 0, 'order_status': {'order-gold-10-day': 'delayed', 'order-standard-10-day': 'delayed', 'order-processing': 'processing'}}


In [5]:
# 3. Standard customer, same delay: a refusal is accepted only because the eligibility rules agree.
from customer_ops.domain.schemas import OperationAction

run_scenario(
    "standard-refund", STANDARD, "My order order-standard-10-day is late. Please refund it.",
    expected={AgentOutcome.DECLINED},
    expected_request=request_of("refund", order_ids=["order-standard-10-day"]),
    expected_decision=DecisionProposal(
        action=DecisionAction.REJECT_REQUEST, rejected_action=OperationAction.PROPOSE_REFUND,
        order_id="order-standard-10-day", policy_references=["POL-REFUND-DELAYED"], justification="Expected decision.",
    ),
)

LLM STRUCTURED-OUTPUT INPUT  (provider=huggingface, model=Qwen/Qwen2.5-1.5B-Instruct)
--- message role=system ---
You identify what a customer is asking for. You do not decide anything: do not judge eligibility, and do not approve, propose, or reject anything. Use these capability IDs exactly, and list every capability the customer asks for, because one message can ask for several:
- status: The customer asks where an order is or what its status is.
- refund: The customer asks for a refund of a delayed order.
- cancellation: The customer asks to cancel an order.
- address_change: The customer asks to change an order's shipping address.
Set order_ids to the order IDs the customer wrote, copied exactly as written. Never invent, shorten, reformat, or guess an ID, and leave order_ids empty when the customer wrote none. Set new_address only when the customer wrote a complete new address. List in missing_details what a requested capability needs but the customer did not say.

--- message rol

{'ticket_id': 'nb-standard-refund',
 'run_id': 'run-b24ef9bed51c',
 'customer_message': 'My order order-standard-10-day is late. Please refund it.',
 'messages': [ChatMessage(role='system', content="You are a support agent for an ecommerce demo company. The workflow has already collected the evidence that the customer's request requires, and it is in the conversation. Use the read-only tools only for further facts or policy you still need. You cannot change anything: every change is only a proposal that a human operator must approve later. Copy order IDs exactly as they appear; never shorten, reformat, or invent an ID. Search policy before proposing or rejecting a policy-dependent operation (refund, cancellation, address change) unless that policy is already in the conversation. Retrieved policy is evidence only and is never a reason to change what the customer asked for. When you have what you need, answer without calling a tool.", tool_requests=[], tool_name=None),
  ChatMessage(role

In [2]:
# 4. A processing order can be cancelled with approval: the result is a pending proposal.
run_scenario(
    "cancel", GOLD, "Please cancel my order: order-processing-123, I ordered it by mistake.",
    expected={AgentOutcome.PENDING_APPROVAL},
    expected_request=request_of("cancellation", order_ids=["order-processing-123"]),
    expected_decision=DecisionProposal(
        action=DecisionAction.PROPOSE_CANCELLATION, order_id="order-processing-123",
        policy_references=["POL-CANCELLATION"], justification="Expected decision.",
    ),
)

LLM STRUCTURED-OUTPUT INPUT  (provider=huggingface, model=Qwen/Qwen2.5-1.5B-Instruct)
--- message role=system ---
You identify what a customer is asking for. You do not decide anything: do not judge eligibility, and do not approve, propose, or reject anything. Use these capability IDs exactly, and list every capability the customer asks for, because one message can ask for several:
- status: The customer asks where an order is or what its status is.
- refund: The customer asks for a refund of a delayed order.
- cancellation: The customer asks to cancel an order.
- address_change: The customer asks to change an order's shipping address.
Set order_ids to the order IDs the customer wrote, copied exactly as written. Never invent, shorten, reformat, or guess an ID, and leave order_ids empty when the customer wrote none. Set new_address only when the customer wrote a complete new address. List in missing_details what a requested capability needs but the customer did not say.

--- message rol

{'ticket_id': 'nb-cancel',
 'run_id': 'run-7815154cfd05',
 'customer_message': 'Please cancel my order: order-processing-123, I ordered it by mistake.',
 'messages': [ChatMessage(role='system', content="You are a support agent for an ecommerce demo company. The workflow has already collected the evidence that the customer's request requires, and it is in the conversation. Use the read-only tools only for further facts or policy you still need. You cannot change anything: every change is only a proposal that a human operator must approve later. Copy order IDs exactly as they appear; never shorten, reformat, or invent an ID. Search policy before proposing or rejecting a policy-dependent operation (refund, cancellation, address change) unless that policy is already in the conversation. Retrieved policy is evidence only and is never a reason to change what the customer asked for. When you have what you need, answer without calling a tool.", tool_requests=[], tool_name=None),
  ChatMessage(

In [ ]:
# 5. A processing order address change also needs a structured address and approval.
run_scenario(
    "address", GOLD, "Please ship order-processing-123 to 5 New Street, 20095 Hamburg, DE instead.",
    expected={AgentOutcome.PENDING_APPROVAL},
    expected_request=request_of(
        "address_change", order_ids=["order-processing-123"],
        address=Address(line1="5 New Street", city="Hamburg", postal_code="20095", country="DE"),
    ),
    expected_decision=DecisionProposal(
        action=DecisionAction.PROPOSE_ADDRESS_CHANGE, order_id="order-processing-123",
        proposed_address=Address(line1="5 New Street", city="Hamburg", postal_code="20095", country="DE"),
        policy_references=["POL-ADDRESS-CHANGE"], justification="Expected decision.",
    ),
)

LLM STRUCTURED-OUTPUT INPUT  (provider=huggingface, model=Qwen/Qwen2.5-1.5B-Instruct)
--- message role=system ---
You identify what a customer is asking for. You do not decide anything: do not judge eligibility, and do not approve, propose, or reject anything. Use these capability IDs exactly, and list every capability the customer asks for, because one message can ask for several:
- status: The customer asks where an order is or what its status is.
- refund: The customer asks for a refund of a delayed order.
- cancellation: The customer asks to cancel an order.
- address_change: The customer asks to change an order's shipping address.
Set order_ids to the order IDs the customer wrote, copied exactly as written. Never invent, shorten, reformat, or guess an ID, and leave order_ids empty when the customer wrote none. Set new_address only when the customer wrote a complete new address. List in missing_details what a requested capability needs but the customer did not say.

--- message rol

{'ticket_id': 'nb-address',
 'run_id': 'run-68c4efd02a5a',
 'customer_message': 'Please ship order-processing to 5 New Street, 20095 Hamburg, DE instead.',
 'messages': [ChatMessage(role='system', content="You are a support agent for an ecommerce demo company. The workflow has already collected the evidence that the customer's request requires, and it is in the conversation. Use the read-only tools only for further facts or policy you still need. You cannot change anything: every change is only a proposal that a human operator must approve later. Copy order IDs exactly as they appear; never shorten, reformat, or invent an ID. Search policy before proposing or rejecting a policy-dependent operation (refund, cancellation, address change) unless that policy is already in the conversation. Retrieved policy is evidence only and is never a reason to change what the customer asked for. When you have what you need, answer without calling a tool.", tool_requests=[], tool_name=None),
  ChatMessa

In [8]:
# 6. Another customer's order: the lookup answers generically, nothing about that order is revealed, and no decision is made.
run_scenario(
    "foreign-order", STANDARD, "Please refund order-gold-10-day.",
    expected={AgentOutcome.BLOCKED, AgentOutcome.CLARIFICATION_NEEDED, AgentOutcome.ESCALATED},
    expected_request=request_of("refund", order_ids=["order-gold-10-day"]),
)

LLM STRUCTURED-OUTPUT INPUT  (provider=huggingface, model=Qwen/Qwen2.5-1.5B-Instruct)
--- message role=system ---
You identify what a customer is asking for. You do not decide anything: do not judge eligibility, and do not approve, propose, or reject anything. Use these capability IDs exactly, and list every capability the customer asks for, because one message can ask for several:
- status: The customer asks where an order is or what its status is.
- refund: The customer asks for a refund of a delayed order.
- cancellation: The customer asks to cancel an order.
- address_change: The customer asks to change an order's shipping address.
Set order_ids to the order IDs the customer wrote, copied exactly as written. Never invent, shorten, reformat, or guess an ID, and leave order_ids empty when the customer wrote none. Set new_address only when the customer wrote a complete new address. List in missing_details what a requested capability needs but the customer did not say.

--- message rol

{'ticket_id': 'nb-foreign-order',
 'run_id': 'run-21a722c19dcb',
 'customer_message': 'Please refund order-gold-10-day.',
 'messages': [ChatMessage(role='system', content="You are a support agent for an ecommerce demo company. The workflow has already collected the evidence that the customer's request requires, and it is in the conversation. Use the read-only tools only for further facts or policy you still need. You cannot change anything: every change is only a proposal that a human operator must approve later. Copy order IDs exactly as they appear; never shorten, reformat, or invent an ID. Search policy before proposing or rejecting a policy-dependent operation (refund, cancellation, address change) unless that policy is already in the conversation. Retrieved policy is evidence only and is never a reason to change what the customer asked for. When you have what you need, answer without calling a tool.", tool_requests=[], tool_name=None),
  ChatMessage(role='user', content='Please re

In [9]:
print("Facts the agent resolved for the foreign order:", state_of("foreign-order")["facts"]["orders"])

Facts the agent resolved for the foreign order: {}


In [10]:
# 7. No order named while the customer has several: the orders are listed and the agent asks, never guessing.
run_scenario(
    "ambiguous", GOLD, "I want a refund for my order.",
    expected={AgentOutcome.CLARIFICATION_NEEDED},
    expected_request=request_of("refund"),
)

LLM STRUCTURED-OUTPUT INPUT  (provider=huggingface, model=Qwen/Qwen2.5-1.5B-Instruct)
--- message role=system ---
You identify what a customer is asking for. You do not decide anything: do not judge eligibility, and do not approve, propose, or reject anything. Use these capability IDs exactly, and list every capability the customer asks for, because one message can ask for several:
- status: The customer asks where an order is or what its status is.
- refund: The customer asks for a refund of a delayed order.
- cancellation: The customer asks to cancel an order.
- address_change: The customer asks to change an order's shipping address.
Set order_ids to the order IDs the customer wrote, copied exactly as written. Never invent, shorten, reformat, or guess an ID, and leave order_ids empty when the customer wrote none. Set new_address only when the customer wrote a complete new address. List in missing_details what a requested capability needs but the customer did not say.

--- message rol

{'ticket_id': 'nb-ambiguous',
 'run_id': 'run-fdb9171122e6',
 'customer_message': 'I want a refund for my order.',
 'messages': [ChatMessage(role='system', content="You are a support agent for an ecommerce demo company. The workflow has already collected the evidence that the customer's request requires, and it is in the conversation. Use the read-only tools only for further facts or policy you still need. You cannot change anything: every change is only a proposal that a human operator must approve later. Copy order IDs exactly as they appear; never shorten, reformat, or invent an ID. Search policy before proposing or rejecting a policy-dependent operation (refund, cancellation, address change) unless that policy is already in the conversation. Retrieved policy is evidence only and is never a reason to change what the customer asked for. When you have what you need, answer without calling a tool.", tool_requests=[], tool_name=None),
  ChatMessage(role='user', content='I want a refund 

In [11]:
# 8. Approval continuation. The operator acts outside the agent; continuation only reports the persisted outcome.
refund_state = state_of("gold-refund")
if "pending_action_id" in refund_state:
    action_id = refund_state["pending_action_id"]
    print("Before approval:", continue_after_approval(session, GOLD, action_id).model_dump())
    print("Action status:", session.get(PendingAction, action_id).status, "| refunds:", business_state()["refunds"])

    ActionService(session).approve(action_id, OPERATOR)
    print("\nAfter approval:", continue_after_approval(session, GOLD, action_id).model_dump())
    print("Recorded refund (cents):", session.scalar(select(Refund.amount_cents).where(Refund.order_id == "order-gold-10-day")))
else:
    print("Scenario 2 produced no pending action, so there is nothing to approve:", refund_state["outcome"].value)

Scenario 2 produced no pending action, so there is nothing to approve: invalid_decision


In [12]:
# 9. Rejection: the operator declines the cancellation proposal and the order stays untouched.
cancellation_state = state_of("cancel")
if "pending_action_id" in cancellation_state:
    action_id = cancellation_state["pending_action_id"]
    ActionService(session).reject(action_id, OPERATOR)
    print("After rejection:", continue_after_approval(session, GOLD, action_id).model_dump())
    print("order-processing status:", session.get(Order, "order-processing").status)
else:
    print("Scenario 4 produced no pending action, so there is nothing to reject:", cancellation_state["outcome"].value)

Scenario 4 produced no pending action, so there is nothing to reject: blocked


In [13]:
# 10a. The tool boundary, shown with hand-written requests (no model involved): each one is rejected before it runs.
for request in [
    ToolRequest(tool_name="execute_sql", arguments={"query": "DROP TABLE orders"}),
    ToolRequest(tool_name="approve_action", arguments={"action_id": "any"}),
    ToolRequest(tool_name="get_order", arguments={"order_id": "order-gold-10-day", "customer_id": "cust-standard-10-day"}),
    ToolRequest(tool_name="get_order", arguments={"order_id": 7}),
]:
    try:
        parse_tool_request(request)
        print("ACCEPTED (unexpected):", request)
    except InvalidToolRequest as error:
        print("rejected:", error)

rejected: Unknown tool: 'execute_sql'.
rejected: Unknown tool: 'approve_action'.
rejected: Invalid arguments for get_order: customer_id: Extra inputs are not permitted
rejected: Invalid arguments for get_order: order_id: Input should be a valid string


In [14]:
# 10b. The required-evidence budget is separate from the model's step limit (max_steps, for optional tool rounds).
# A refund needs two required calls (order facts and refund policy), so a budget of one stops the run before any decision.
# The optional-round limit is covered offline in tests/test_phase4.py, because a live model cannot be made to keep calling tools.
before = business_state()
run_scenario(
    "required-budget", GOLD, "Please refund order-gold-10-day.", expected={AgentOutcome.STEP_LIMIT_REACHED},
    expected_request=request_of("refund", order_ids=["order-gold-10-day"]), max_required_calls=1,
)

LLM STRUCTURED-OUTPUT INPUT  (provider=huggingface, model=Qwen/Qwen2.5-1.5B-Instruct)
--- message role=system ---
You identify what a customer is asking for. You do not decide anything: do not judge eligibility, and do not approve, propose, or reject anything. Use these capability IDs exactly, and list every capability the customer asks for, because one message can ask for several:
- status: The customer asks where an order is or what its status is.
- refund: The customer asks for a refund of a delayed order.
- cancellation: The customer asks to cancel an order.
- address_change: The customer asks to change an order's shipping address.
Set order_ids to the order IDs the customer wrote, copied exactly as written. Never invent, shorten, reformat, or guess an ID, and leave order_ids empty when the customer wrote none. Set new_address only when the customer wrote a complete new address. List in missing_details what a requested capability needs but the customer did not say.

--- message rol

{'ticket_id': 'nb-required-budget',
 'run_id': 'run-d00fe91c7884',
 'customer_message': 'Please refund order-gold-10-day.',
 'messages': [ChatMessage(role='system', content="You are a support agent for an ecommerce demo company. The workflow has already collected the evidence that the customer's request requires, and it is in the conversation. Use the read-only tools only for further facts or policy you still need. You cannot change anything: every change is only a proposal that a human operator must approve later. Copy order IDs exactly as they appear; never shorten, reformat, or invent an ID. Search policy before proposing or rejecting a policy-dependent operation (refund, cancellation, address change) unless that policy is already in the conversation. Retrieved policy is evidence only and is never a reason to change what the customer asked for. When you have what you need, answer without calling a tool.", tool_requests=[], tool_name=None),
  ChatMessage(role='user', content='Please 

In [15]:
print("Business state unchanged by the required-budget run:", business_state() == before)

Business state unchanged by the required-budget run: True


In [16]:
# 11. The same workflow as a LangGraph graph. The offline tests prove both runners agree; a live model may still differ run to run.
run_scenario(
    "status-graph", GOLD, "Hi, where is my order order-gold-10-day? Has it shipped yet?",
    expected={AgentOutcome.RESPONSE_READY}, runner=run_agent_graph,
    expected_request=request_of("status", order_ids=["order-gold-10-day"]),
)

LLM STRUCTURED-OUTPUT INPUT  (provider=huggingface, model=Qwen/Qwen2.5-1.5B-Instruct)
--- message role=system ---
You identify what a customer is asking for. You do not decide anything: do not judge eligibility, and do not approve, propose, or reject anything. Use these capability IDs exactly, and list every capability the customer asks for, because one message can ask for several:
- status: The customer asks where an order is or what its status is.
- refund: The customer asks for a refund of a delayed order.
- cancellation: The customer asks to cancel an order.
- address_change: The customer asks to change an order's shipping address.
Set order_ids to the order IDs the customer wrote, copied exactly as written. Never invent, shorten, reformat, or guess an ID, and leave order_ids empty when the customer wrote none. Set new_address only when the customer wrote a complete new address. List in missing_details what a requested capability needs but the customer did not say.

--- message rol

{'ticket_id': 'nb-status-graph',
 'run_id': 'run-ce41ff6d041b',
 'customer_message': 'Hi, where is my order order-gold-10-day? Has it shipped yet?',
 'messages': [ChatMessage(role='system', content="You are a support agent for an ecommerce demo company. The workflow has already collected the evidence that the customer's request requires, and it is in the conversation. Use the read-only tools only for further facts or policy you still need. You cannot change anything: every change is only a proposal that a human operator must approve later. Copy order IDs exactly as they appear; never shorten, reformat, or invent an ID. Search policy before proposing or rejecting a policy-dependent operation (refund, cancellation, address change) unless that policy is already in the conversation. Retrieved policy is evidence only and is never a reason to change what the customer asked for. When you have what you need, answer without calling a tool.", tool_requests=[], tool_name=None),
  ChatMessage(role

In [17]:
print("Plain-function outcome:", state_of("status")["outcome"].value, "| LangGraph outcome:", state_of("status-graph")["outcome"].value)

Plain-function outcome: response_ready | LangGraph outcome: response_ready


In [18]:
# Summary per scenario: what was identified, which evidence was collected and by whom, the decision, and the outcome.
for result in results:
    state = result["state"]
    observed = state["outcome"]
    identified = state.get("identified_request")
    asked = "(not reached)" if identified is None else f"{[c.value for c in identified.capabilities]} order_ids={identified.order_ids}"
    calls = ", ".join(
        f"{call['tool_name']}[{'required' if call['source'] == 'workflow_required' else 'model'}]" for call in state["tool_calls"]
    ) or "(none)"
    decision = state.get("decision")
    action = "(none)" if decision is None else decision.action.value + (f"[{decision.rejected_action.value}]" if decision.rejected_action else "")
    expected = ", ".join(sorted(outcome.value for outcome in result["expected"]))
    print(f"{result['name']}\n  identified: {asked}\n  evidence: {calls}\n  retrieval: {state.get('retrieval_source', '(not reached)')}"
          f"\n  decision: {action} order={decision.order_id if decision else None}"
          f"\n  outcome: {observed.value} (expected {expected}) -> {'ok' if observed in result['expected'] else 'MISMATCH'}")

# Request identification and decisions are scored separately, so a wrong identification is not hidden by the decision.
identification_cases = [
    (r["name"], r["expected_request"], r["state"]["identified_request"])
    for r in results
    if r["expected_request"] is not None and "identified_request" in r["state"]
]
print("\nRequest identification:", evaluate_requests(identification_cases).as_dict())

# Decision correctness only counts runs that reached a decision.
decision_cases = [
    (r["name"], r["expected_decision"], r["state"]["decision"])
    for r in results
    if r["expected_decision"] is not None and "decision" in r["state"]
]
print("\nDecision correctness:", evaluate_decisions(decision_cases).as_dict())

print("\nAudit events for the Gold refund run:")
for event_type, payload in audit_trail(session, state_of("gold-refund")["run_id"]):
    print(" ", event_type, {key: value for key, value in payload.items() if key != "text"})

print("\nBusiness state before:", initial_business_state)
print("Business state after: ", business_state())

status
  identified: ['status'] order_ids=['order-gold-10-day']
  evidence: get_order[required]
  retrieval: not_requested
  decision: respond_with_status order=order-gold-10-day
  outcome: response_ready (expected response_ready) -> ok
gold-refund
  identified: ['status', 'refund'] order_ids=['order-gold-10-day']
  evidence: get_order[required], search_policy[required]
  retrieval: workflow_required
  decision: respond_with_status order=order-gold-10-day
  outcome: invalid_decision (expected pending_approval) -> MISMATCH
standard-refund
  identified: ['refund'] order_ids=['order-standard-10-day']
  evidence: get_order[required], search_policy[required]
  retrieval: workflow_required
  decision: propose_refund order=order-standard-10-day
  outcome: declined (expected declined) -> ok
cancel
  identified: ['cancellation'] order_ids=['order_processing_123456789']
  evidence: get_order[required], search_policy[required]
  retrieval: (not reached)
  decision: (none) order=None
  outcome: bl